In [ ]:
import pathlib
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Working with Results

After completing an analysis with *PedPy*, you may want to:

- **Access specific columns** of the computed results
- **Combine** multiple result DataFrames for further analysis
- **Save** results to disk

First, we set up the data and compute some example results (see [Analysis Setup](measurement_setup) and [Analysis](analysis) for details):


In [ ]:
from pedpy import (
    FRAME_COL,
    ID_COL,
    MeasurementArea,
    MeasurementLine,
    SpeedCalculation,
    TrajectoryUnit,
    WalkableArea,
    compute_individual_speed,
    compute_individual_voronoi_polygons,
    compute_profiles,
    compute_voronoi_density,
    load_trajectory,
)

walkable_area = WalkableArea(
    [
        (3.5, -2),
        (3.5, 8),
        (-3.5, 8),
        (-3.5, -2),
    ],
    obstacles=[
        [
            (-0.7, -1.1),
            (-0.25, -1.1),
            (-0.25, -0.15),
            (-0.4, 0.0),
            (-2.8, 0.0),
            (-2.8, 6.7),
            (-3.05, 6.7),
            (-3.05, -0.3),
            (-0.7, -0.3),
            (-0.7, -1.0),
        ],
        [
            (0.25, -1.1),
            (0.7, -1.1),
            (0.7, -0.3),
            (3.05, -0.3),
            (3.05, 6.7),
            (2.8, 6.7),
            (2.8, 0.0),
            (0.4, 0.0),
            (0.25, -0.15),
            (0.25, -1.1),
        ],
    ],
)

measurement_area = MeasurementArea([(-0.4, 0.5), (0.4, 0.5), (0.4, 1.3), (-0.4, 1.3)])
measurement_line = MeasurementLine([(0.4, 0), (-0.4, 0)])

traj = load_trajectory(
    trajectory_file=pathlib.Path("demo-data/bottleneck/040_c_56_h-.txt"),
    default_unit=TrajectoryUnit.METER,
)

individual = compute_individual_voronoi_polygons(traj_data=traj, walkable_area=walkable_area)
density_voronoi, intersecting = compute_voronoi_density(
    individual_voronoi_data=individual, measurement_area=measurement_area
)
individual_speed = compute_individual_speed(
    traj_data=traj,
    frame_step=5,
    speed_calculation=SpeedCalculation.BORDER_SINGLE_SIDED,
)

## Access specific columns

In order to access a specific column of the computed results or the {class}`trajectory data<trajectory_data.TrajectoryData>` *PedPy* provides all the column names for access.
It is highly encouraged to use these identifiers instead of using the column names directly, as the identifier would be updated, if they change. 
A list of all identifiers can be found in the [API reference](api_column_identifier).


In [ ]:
from pedpy import FRAME_COL, ID_COL, POINT_COL

# Access with column identifier
traj.data[[ID_COL, FRAME_COL, POINT_COL]]

In [ ]:
# Access with column names
traj.data[["id", "frame", "point"]]

## Combine multiple DataFrames

From the analysis we have one {class}`~pandas.DataFrame` containing the {class}`trajectory data<trajectory_data.TrajectoryData>`:


In [ ]:
traj.data

and one containing the individual Voronoi data:


In [ ]:
individual

As both have the columns 'id' and 'frame' in common, these can be used to merge the dataframes.
See {func}`pandas.merge` and {meth}`pandas.DataFrame.merge` for more information on how {class}`DataFrames <pandas.DataFrame>` can be merged.


In [ ]:
data_with_voronoi_cells = traj.data.merge(intersecting, on=[ID_COL, FRAME_COL])
data_with_voronoi_cells

In [ ]:
from pedpy import SPEED_COL

data_with_voronoi_cells_speed = data_with_voronoi_cells.merge(
    individual_speed[[ID_COL, FRAME_COL, SPEED_COL]], on=[ID_COL, FRAME_COL]
)
data_with_voronoi_cells_speed

## Save in files

For preserving the results on the disk, the results need to be saved on the disk.
This can be done with the build-in functions from *Pandas*.


### Create directories to store the results

First we create a directory, where we want to save the results.
This step is optional!


In [ ]:
pathlib.Path("results_introduction/profiles/speed").mkdir(parents=True, exist_ok=True)
pathlib.Path("results_introduction/profiles/density").mkdir(parents=True, exist_ok=True)

results_directory = pathlib.Path("results_introduction")

### Save Pandas DataFrame (result from everything but profiles) as csv

Now, a {class}`~pandas.DataFrame` can be saved as `csv` with:


In [ ]:
import csv

from pedpy import (
    DENSITY_COL,
    INTERSECTION_COL,
    POLYGON_COL,
    X_COL,
    Y_COL,
)

with open(results_directory / "individual_result.csv", "w") as individual_output_file:
    individual_output_file.write(f"#framerate:\t{traj.frame_rate}\n\n")
    data_with_voronoi_cells_speed[
        [
            ID_COL,
            FRAME_COL,
            X_COL,
            Y_COL,
            DENSITY_COL,
            SPEED_COL,
            POLYGON_COL,
            INTERSECTION_COL,
        ]
    ].to_csv(
        individual_output_file,
        mode="a",
        header=True,
        sep="\t",
        index_label=False,
        index=False,
        quoting=csv.QUOTE_NONNUMERIC,
    )

### Save numpy arrays (result from profiles) as txt

The profiles are returned as Numpy arrays, which also provide a build-in save function, which allows to save the arrays as txt format:


In [ ]:
from pedpy import Cutoff, DensityMethod, SpeedMethod

individual_cutoff = compute_individual_voronoi_polygons(
    traj_data=traj,
    walkable_area=walkable_area,
    cut_off=Cutoff(radius=0.8, quad_segments=3),
)

min_frame_profiles = 250
max_frame_profiles = 400
grid_size = 0.4

density_profiles, speed_profiles = compute_profiles(
    data=pd.merge(
        individual_cutoff[individual_cutoff.frame.between(min_frame_profiles, max_frame_profiles)],
        individual_speed[individual_speed.frame.between(min_frame_profiles, max_frame_profiles)],
        on=[ID_COL, FRAME_COL],
    ),
    walkable_area=walkable_area.polygon,
    grid_size=grid_size,
    speed_method=SpeedMethod.ARITHMETIC,
    density_method=DensityMethod.VORONOI,
)

In [ ]:
results_directory_density = results_directory / "profiles/density"
results_directory_speed = results_directory / "profiles/speed"

for i in range(len(range(min_frame_profiles, min_frame_profiles + 10))):
    frame = min_frame_profiles + i
    np.savetxt(
        results_directory_density / f"density_frame_{frame:05d}.txt",
        density_profiles[i],
    )
    np.savetxt(
        results_directory_speed / f"speed_frame_{frame:05d}.txt",
        speed_profiles[i],
    )